# Notebook 11 — Discrete Global Grid Systems as Tile Identifiers

The current pipeline quantises projected coordinates to 250 m Web Mercator squares:

```python
qx = round(x / 250)
qy = round(y / 250)
```

This notebook explores replacing that step with **H3**, Uber's open-source
hexagonal DGGS (Discrete Global Grid System). H3 provides:

- **Equal-area cells** at each resolution (unlike Web Mercator, which distorts
  poleward)
- **Hierarchical addressing** — coarser resolutions nest cleanly inside finer ones
- **Semantic interoperability** — cell IDs are a standard spatial reference used
  across analytics platforms

The OGC DGGS standard defines a DGGS as a *hierarchical tessellation of cells
for organizing and analyzing global spatial data with referencing by identifiers
with structured geometry.* H3 implements these principles with hexagonal cells
and a 64-bit integer cell ID.

**Important:** DGGS provides the *spatial reference layer* only. The PRP + AEAD
encryption architecture from NB03–NB04 remains the *privacy/security layer*.
DGGS does not itself provide privacy or encryption.

## Setup

In [1]:
import h3
import pandas as pd
import numpy as np
import folium
import plotly.express as px
import math

# Broadwick Street pump — anchor point for all demonstrations
CENTER_LAT, CENTER_LON = 51.513341, -0.136668

deaths = pd.read_csv('data/cholera_deaths.csv')   # FID, DEATHS, LON, LAT
pumps  = pd.read_csv('data/pumps.csv')             # FID, LON, LAT, Street

print(f'h3 version: {h3.__version__}')
print(f'Deaths: {len(deaths)} records, Pumps: {len(pumps)} records')

h3 version: 4.4.1
Deaths: 250 records, Pumps: 8 records


## Part 1 — Choosing a Resolution

H3 has 16 resolutions (0 = globe, 15 = ~1 m²). The table below shows area
and average edge length for resolutions near the current 250 m tile.
The **250 m Web Mercator tile** has area ≈ 0.0625 km² at the equator.

In [2]:
rows = []
for res in range(7, 12):
    cell = h3.latlng_to_cell(CENTER_LAT, CENTER_LON, res)
    area  = h3.cell_area(cell, unit='km^2')
    edge  = h3.average_hexagon_edge_length(res, unit='m')
    rows.append({'Resolution': res,
                 'Avg edge (m)': round(edge),
                 'Cell area (km²)': round(area, 5),
                 'Example cell ID': cell})

res_df = pd.DataFrame(rows)
print(res_df.to_string(index=False))
print()
print('Web Mercator 250 m tile area: 0.06250 km²')
print('Closest H3 resolution: 9  (avg edge 201 m, area 0.094 km²)')

 Resolution  Avg edge (m)  Cell area (km²) Example cell ID
          7          1406          4.61548 87195da49ffffff
          8           531          0.65924 88195da499fffff
          9           201          0.09418 89195da498fffff
         10            76          0.01345 8a195da498f7fff
         11            29          0.00192 8b195da498f1fff

Web Mercator 250 m tile area: 0.06250 km²
Closest H3 resolution: 9  (avg edge 201 m, area 0.094 km²)


## Part 2 — Encoding Cholera Deaths to H3 Cells

`h3.latlng_to_cell(lat, lon, resolution)` converts any (lat, lon) to the
H3 cell ID containing that point. The cell ID is a 15-hex-digit string
representing a 64-bit integer.

Resolution 9 is used throughout this notebook as the primary comparison
with the current 250 m Web Mercator grid.

In [3]:
RES = 9

deaths['h3_cell'] = deaths.apply(
    lambda r: h3.latlng_to_cell(r.LAT, r.LON, RES), axis=1)

n_cells   = deaths['h3_cell'].nunique()
n_records = len(deaths)

print(f'Resolution {RES}: {n_cells} unique H3 cells for {n_records} death records')
print(f'Avg deaths per occupied cell: {n_records / n_cells:.2f}')
print()
print('Example records:')
print(deaths[['FID', 'LAT', 'LON', 'DEATHS', 'h3_cell']].head(6).to_string(index=False))

Resolution 9: 5 unique H3 cells for 250 death records
Avg deaths per occupied cell: 50.00

Example records:
 FID       LAT       LON  DEATHS         h3_cell
   0 51.513418 -0.137930       3 89195da498fffff
   1 51.513361 -0.137883       2 89195da498fffff
   2 51.513317 -0.137853       1 89195da498fffff
   3 51.513262 -0.137812       1 89195da498fffff
   4 51.513204 -0.137767       4 89195da498fffff
   5 51.513184 -0.137537       2 89195da498fffff


## Part 3 — Cell Centre and Intra-cell Residual

Just as the current pipeline stores the sub-tile residual `(rx, ry)` in
projected metres, a DGGS pipeline would store the **intra-cell residual**
as the offset from the H3 cell centre in metres.

The residual plays the same cryptographic role: it is AEAD-encrypted and
bound to the (permuted) cell ID via associated data.

In [4]:
def haversine_m(lat1, lon1, lat2, lon2):
    R = 6_371_000.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi  = math.radians(lat2 - lat1)
    dlam  = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2
    return 2 * R * math.asin(math.sqrt(a))

def cell_residual_m(lat, lon, cell):
    clat, clon = h3.cell_to_latlng(cell)
    dlat_m = haversine_m(lat, lon, clat, lon)
    dlon_m = haversine_m(lat, lon, lat,  clon)
    if lat  < clat: dlat_m = -dlat_m
    if lon  < clon: dlon_m = -dlon_m
    return dlat_m, dlon_m

deaths[['res_lat_m', 'res_lon_m']] = deaths.apply(
    lambda r: pd.Series(cell_residual_m(r.LAT, r.LON, r.h3_cell)), axis=1)

max_res = deaths[['res_lat_m', 'res_lon_m']].abs().max()
print('Max intra-cell residual (m):')
print(f'  lat axis: {max_res.res_lat_m:.1f} m')
print(f'  lon axis: {max_res.res_lon_m:.1f} m')
print()
print('Compare: Web Mercator 250 m tile → max residual ±125 m per axis')
print(f'H3 res {RES} avg edge {h3.average_hexagon_edge_length(RES, unit="m"):.0f} m → max residual ≈ ±{h3.average_hexagon_edge_length(RES, unit="m")/2:.0f} m per axis')

Max intra-cell residual (m):
  lat axis: 194.2 m
  lon axis: 170.5 m

Compare: Web Mercator 250 m tile → max residual ±125 m per axis
H3 res 9 avg edge 201 m → max residual ≈ ±100 m per axis


## Part 4 — H3 Cell Boundaries on a Folium Map

Each H3 cell has a hexagonal boundary that can be rendered as a GeoJSON
polygon. Below, occupied cells are coloured by death count; unoccupied
cells are not drawn. This is the display tier view — only approximate
cell-level positions, not precise (lat, lon) values.

In [5]:
cell_deaths = deaths.groupby('h3_cell')['DEATHS'].sum().reset_index()
max_d = cell_deaths['DEATHS'].max()

m = folium.Map(location=[CENTER_LAT, CENTER_LON], zoom_start=15,
               tiles='cartodbpositron')

for _, row in cell_deaths.iterrows():
    boundary = h3.cell_to_boundary(row['h3_cell'])
    # h3 returns (lat, lon) pairs; folium expects the same
    intensity = row['DEATHS'] / max_d
    r = int(255 * intensity)
    g = int(50  * (1 - intensity))
    b = int(50  * (1 - intensity))
    colour = f'#{r:02x}{g:02x}{b:02x}'
    folium.Polygon(
        locations=boundary,
        color='#555555', weight=1,
        fill=True, fill_color=colour, fill_opacity=0.55,
        tooltip=f"Cell: {row['h3_cell']}  Deaths: {int(row['DEATHS'])}"
    ).add_to(m)

# Overlay pump markers
for _, p in pumps.iterrows():
    folium.CircleMarker(
        location=[p.LAT, p.LON], radius=6,
        color='blue', fill=True, fill_color='blue', fill_opacity=0.8,
        tooltip=p.Street
    ).add_to(m)

m

## Part 5 — Multi-resolution Privacy Levels

Because H3 is hierarchical, you can choose the resolution at encode time
to trade utility for disclosure risk. A finer resolution reveals more
location precision; a coarser resolution provides a larger anonymisation
zone — at the cost of less useful spatial aggregation.

The Broadwick Street pump cell at three resolutions:

In [6]:
hdr = f"{'Resolution':>12}  {'Cell ID':>16}  {'Area (km2)':>12}  {'Avg edge (m)':>14}"
print(hdr)
print('-' * 60)
for res in [7, 8, 9]:
    cell = h3.latlng_to_cell(CENTER_LAT, CENTER_LON, res)
    area = h3.cell_area(cell, unit='km^2')
    edge = h3.average_hexagon_edge_length(res, unit='m')
    print(f'{res:>12}  {cell:>16}  {area:>12.5f}  {edge:>14.0f}')
print()
# Verify containment (parent–child relationship)
cell9 = h3.latlng_to_cell(CENTER_LAT, CENTER_LON, 9)
cell8 = h3.latlng_to_cell(CENTER_LAT, CENTER_LON, 8)
cell7 = h3.latlng_to_cell(CENTER_LAT, CENTER_LON, 7)
print('Containment check (child cell → parent cell):')
print(f'  res9 parent at res8: {h3.cell_to_parent(cell9, 8)}')
print(f'  res8 matches res8 cell: {h3.cell_to_parent(cell9, 8) == cell8}')
print(f'  res8 parent at res7: {h3.cell_to_parent(cell8, 7)}')
print(f'  res7 matches res7 cell: {h3.cell_to_parent(cell8, 7) == cell7}')

  Resolution           Cell ID    Area (km2)    Avg edge (m)
------------------------------------------------------------
           7   87195da49ffffff       4.61548            1406
           8   88195da499fffff       0.65924             531
           9   89195da498fffff       0.09418             201

Containment check (child cell → parent cell):
  res9 parent at res8: 88195da499fffff
  res8 matches res8 cell: True
  res8 parent at res7: 87195da49ffffff
  res7 matches res7 cell: True


In [7]:
# Folium map showing three nested cell boundaries
m2 = folium.Map(location=[CENTER_LAT, CENTER_LON], zoom_start=14,
                tiles='cartodbpositron')

res_styles = {
    7: ('Resolution 7 (~4.6 km²)', '#1a6a1a', 0.15),
    8: ('Resolution 8 (~0.66 km²)', '#1a4a9a', 0.20),
    9: ('Resolution 9 (~0.094 km²)', '#9a1a1a', 0.30),
}
for res, (label, colour, opacity) in res_styles.items():
    cell = h3.latlng_to_cell(CENTER_LAT, CENTER_LON, res)
    boundary = h3.cell_to_boundary(cell)
    folium.Polygon(
        locations=boundary,
        color=colour, weight=2,
        fill=True, fill_color=colour, fill_opacity=opacity,
        tooltip=label
    ).add_to(m2)

folium.CircleMarker(
    location=[CENTER_LAT, CENTER_LON], radius=6,
    color='black', fill=True, fill_color='red', fill_opacity=1.0,
    tooltip='Broadwick Street pump'
).add_to(m2)

m2

## Part 6 — Equal-area Advantage over Web Mercator

Web Mercator tiles have *constant index size in projected metres* but their
geographic area grows rapidly toward the poles. A 250 m tile at latitude 51°
(London) is already ~20% smaller in degrees² than the same tile at the equator.

H3 cells maintain approximately equal area at each resolution regardless of
latitude, making density comparisons (deaths per cell) meaningful without
correction factors.

In [8]:
# Web Mercator: how many m² per degree-square at a given latitude?
# Mercator scale factor = 1/cos(lat) — tile area in km² at equator vs lat
latitudes = np.linspace(0, 75, 200)
# 250 m Web Mercator tile: 0.0625 km² at equator
# Geographic area shrinks as 1/sec²(lat) = cos²(lat)
merc_area = 0.0625 * np.cos(np.radians(latitudes))**2

fig = px.line(
    x=latitudes, y=merc_area,
    labels={'x': 'Latitude (°)', 'y': 'Geographic area (km²)'},
    title='250 m Web Mercator tile: geographic area vs latitude'
)
fig.add_hline(
    y=h3.average_hexagon_area(9, unit='km^2'),
    line_dash='dash', line_color='red',
    annotation_text='H3 res 9 (constant ≈ 0.094 km²)',
    annotation_position='bottom right'
)
fig.update_layout(height=380)
fig.show()

## Part 7 — Permuting H3 Cell IDs with a Keyed PRF

The current Feistel PRP permutes integer pairs `(qx, qy)`. With H3, the
natural input is a **single 64-bit integer** cell ID.

A keyed BLAKE2s digest over the cell ID produces a pseudorandom-looking
shuffled index. For a true bijection over the H3 cell namespace at a
given resolution you would use format-preserving encryption (FPE); for
a demonstration, a keyed hash gives the same privacy intuition.

Below, each cell ID in the Soho dataset is mapped to a shuffled integer:
the shuffled value is what would be stored in the primary record in place
of the real cell ID.

In [9]:
import hashlib, struct

def prf_cell(cell_id_str: str, prp_key: bytes) -> int:
    'Keyed BLAKE2s over the H3 cell ID → 64-bit pseudorandom index.'
    cell_int = h3.str_to_int(cell_id_str)
    digest = hashlib.blake2s(
        struct.pack('>Q', cell_int), key=prp_key, digest_size=8
    ).digest()
    return struct.unpack('>Q', digest)[0]

prp_key = bytes.fromhex('deadbeef' * 8)  # 32-byte demo key

sample = deaths[['FID', 'h3_cell']].drop_duplicates('h3_cell').head(6).copy()
sample['cell_int']     = sample['h3_cell'].apply(h3.str_to_int)
sample['shuffled_int'] = sample['h3_cell'].apply(lambda c: prf_cell(c, prp_key))

print(sample[['FID', 'h3_cell', 'cell_int', 'shuffled_int']].to_string(index=False))
print()
print('The shuffled_int is stored in the primary record.')
print('Without prp_key, the real cell_id (and thus location) cannot be recovered.')

 FID         h3_cell           cell_int         shuffled_int
   0 89195da498fffff 617439388696051711 10353357054284792634
  20 89195da498bffff 617439388695789567  7753651241798360741
  22 89195da4913ffff 617439388687925247  6590292215996902045
 143 89195da4987ffff 617439388695527423  4615379003318919991
 149 89195da4983ffff 617439388695265279  7344158508096560050

The shuffled_int is stored in the primary record.
Without prp_key, the real cell_id (and thus location) cannot be recovered.


## Part 8 — Adapted Pipeline

With DGGS replacing the Web Mercator grid, the four-step pipeline becomes:

| Step | Current | DGGS variant |
|------|---------|-------------|
| **Project** | `_project(lat, lon)` → (x, y) metres | not required — H3 operates on (lat, lon) directly |
| **Snap** | `qx = round(x/250)` | `cell_id = h3.latlng_to_cell(lat, lon, res)` |
| **Shuffle** | Feistel PRP on (qx, qy) | Keyed PRF / FPE on 64-bit cell ID |
| **Lock** | ChaCha20-Poly1305 on (rx, ry) in metres | ChaCha20-Poly1305 on (Δlat, Δlon) in metres from cell centre |
| **Wobble** | BLAKE2s jitter within tile | BLAKE2s jitter within H3 cell boundary |

The AEAD associated data would bind ciphertext to the (shuffled) cell ID
and resolution level, preventing cross-cell replay attacks.

**No change** to the key derivation (`_derive_keys`), AEAD construction
(`_build_ad`, `_AEAD`), or jitter mechanics — these are cell-agnostic.

## Part 9 — Security Considerations

DGGS does not provide privacy on its own. The same threats apply:

| Asset | Must be protected | Why |
|-------|-------------------|-----|
| Real cell ID | Yes — with PRP/PRF | Reveals ≤250 m bin |
| Intra-cell residual | Yes — with AEAD | Reveals exact position |
| Resolution level | Protect or treat as public | Reveals precision tier |
| Semantic labels (outbreak zone, hospital) | Yes | Quasi-identifier |
| prp_key | Secret | Needed to reverse cell shuffle |
| aead_key | Secret | Needed to decrypt residual |
| jitter_key | Display tier only | Does not expose precise position |

**Additional DGGS-specific risks:**

- H3 cell boundaries are public. An attacker who recovers the shuffled cell
  ID and has `prp_key` can immediately determine the privacy zone — the same
  threat as recovering `(qxp, qyp)` in the current pipeline.
- If multiple resolutions are used, leaking the resolution level tells an
  attacker which privacy tier was applied per record.
- H3 pentagonal cells (12 per resolution) have slightly different area;
  records in pentagonal cells could be distinguishable by residual range.

## Part 10 — Summary Comparison

| Property | Web Mercator 250 m grid | H3 resolution 9 |
|----------|------------------------|----------------|
| Cell shape | Square | Hexagonal |
| Cell area at London (51°N) | ≈ 0.041 km² (distorted) | ≈ 0.094 km² (equal-area) |
| Avg edge length | 250 m | ≈ 201 m |
| Cell ID type | Integer pair (qx, qy) | 64-bit integer |
| Hierarchical | No | Yes (resolutions 0–15) |
| OGC standard | No | Follows DGGS principles |
| Projection required | Yes (Web Mercator) | No — operates on (lat, lon) |
| Crypto layer change | — | Minimal: PRF replaces Feistel pair PRP |
| Equal-area density maps | No | Yes |
| Python library | stdlib only | `h3-py` (Apache 2.0) |

## References

- Open Geospatial Consortium. (2020). *OGC Abstract Specification Topic 21: Discrete
  Global Grid Systems — Part 1 Core Reference system and Operations and Equal Area
  Earth Reference System.* OGC 20-040r3.
  https://www.ogc.org/standards/dggs/

- Brodsky, I., & Others. (2018). *H3: Uber's Hexagonal Hierarchical Spatial Index.*
  https://eng.uber.com/h3/

- Lin, Y. (2023). Geo-indistinguishable masking: enhancing privacy protection in
  spatial point mapping. *Cartography and Geographic Information Science.*
  https://doi.org/10.1080/15230406.2023.2267967

- Snow, J. (1855). *On the Mode of Communication of Cholera* (2nd ed.). John Churchill.

## Glossary

| Term | Definition |
|------|------------|
| DGGS | Discrete Global Grid System — a hierarchical tessellation of the Earth's surface into cells with unique identifiers |
| H3 | Uber's open-source hexagonal DGGS library; cell IDs are 64-bit integers |
| resolution | H3 precision level (0 = globe, 15 = ~1 m²); resolution 9 ≈ 201 m edge length |
| cell ID | 64-bit integer (or 15-hex-char string) uniquely identifying one H3 cell |
| equal-area | Property where all cells at a given resolution have the same geographic area |
| intra-cell residual | (lat, lon) offset from the H3 cell centre; analogous to (rx, ry) in the current pipeline |
| format-preserving encryption (FPE) | Encryption that maps a plaintext in a given domain to a ciphertext in the same domain |
| keyed PRF | Pseudorandom function taking a key and input; used here to shuffle cell IDs |
| hierarchical containment | Property that every H3 cell at resolution r is fully contained in one cell at resolution r−1 |
| Web Mercator distortion | Area inflation near the poles caused by the Mercator cylindrical projection |